# **Lab 7(B) - Mathematical and Statistical Filters**

In [ ]:
import cv2
import numpy as np

from IPython.display import Image, display
from matplotlib import pyplot as plt

plt.rc('text', usetex=False)
np.random.seed(7)


In [ ]:
# ========== ( Adding noise to an Image ) =================

# 0. Read an Image
imgPath = 'noise.jpg'
img = cv2.imread(imgPath, 0)

if img is None:
    raise FileNotFoundError("noise.jpg was not found. Put noise.jpg in the same folder as this notebook.")

# 1. Function that handles all noise algorithms
def add_noise(image, noise_type):
    row, col = image.shape

    if noise_type == "gaussian":
        mean = 0
        var = 0.01
        sigma = var ** 0.5
        gauss = np.random.normal(mean, sigma, (row, col))
        noisy = image / 255.0 + gauss
        return np.clip(noisy * 255, 0, 255).astype(np.uint8)

    elif noise_type == "salt_pepper":
        s_vs_p = 0.5
        amount = 0.04
        noisy = np.copy(image)

        num_salt = int(np.ceil(amount * image.size * s_vs_p))
        coords = [np.random.randint(0, i, num_salt) for i in image.shape]
        noisy[tuple(coords)] = 255

        num_pepper = int(np.ceil(amount * image.size * (1.0 - s_vs_p)))
        coords = [np.random.randint(0, i, num_pepper) for i in image.shape]
        noisy[tuple(coords)] = 0

        return noisy

    elif noise_type == "exponential":
        expo = np.random.exponential(scale=10.0, size=(row, col))
        noisy = image.astype(np.float64) + expo
        return np.clip(noisy, 0, 255).astype(np.uint8)

    elif noise_type == "uniform":
        low = -20
        high = 20
        uni = np.random.uniform(low, high, (row, col))
        noisy = image.astype(np.float64) + uni
        return np.clip(noisy, 0, 255).astype(np.uint8)

    else:
        raise ValueError("Unknown noise type")

# 2. Generate Noisy Images
noise_gauss = add_noise(img, "gaussian")
noise_sp = add_noise(img, "salt_pepper")
noise_expo = add_noise(img, "exponential")
noise_uni = add_noise(img, "uniform")

# 3. Display in 2x2 Grid
plt.figure(figsize=(12, 10))

plt.subplot(221)
plt.imshow(noise_gauss, cmap='gray')
plt.title('Gaussian Noise')
plt.axis('off')

plt.subplot(222)
plt.imshow(noise_sp, cmap='gray')
plt.title('Salt & Pepper Noise')
plt.axis('off')

plt.subplot(223)
plt.imshow(noise_expo, cmap='gray')
plt.title('Exponential Noise')
plt.axis('off')

plt.subplot(224)
plt.imshow(noise_uni, cmap='gray')
plt.title('Uniform Noise')
plt.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# ========== ( Arithmetic Mean Filter - Built-in OpenCV ) =================

# Apply Arithmetic Mean Filter using cv2.blur
arithmetic_mean_builtin = cv2.blur(noise_gauss, (3, 3))

plt.figure(figsize=(10, 5))

plt.subplot(121)
plt.imshow(noise_gauss, cmap='gray')
plt.title('Gaussian Noisy Image')
plt.axis('off')

plt.subplot(122)
plt.imshow(arithmetic_mean_builtin, cmap='gray')
plt.title('Arithmetic Mean - Built-in')
plt.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# ========== ( Arithmetic Mean Filter - Manual Kernel ) =================

kernel_size = 3
kernel = np.ones((kernel_size, kernel_size), np.float32) / (kernel_size * kernel_size)

arithmetic_mean_manual = cv2.filter2D(noise_gauss, -1, kernel)

plt.figure(figsize=(10, 5))

plt.subplot(121)
plt.imshow(noise_gauss, cmap='gray')
plt.title('Gaussian Noisy Image')
plt.axis('off')

plt.subplot(122)
plt.imshow(arithmetic_mean_manual, cmap='gray')
plt.title('Arithmetic Mean - Manual')
plt.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# Compare built-in and manual Arithmetic Mean Filter

plt.figure(figsize=(15, 5))

plt.subplot(131)
plt.imshow(noise_gauss, cmap='gray')
plt.title('Noisy Image')
plt.axis('off')

plt.subplot(132)
plt.imshow(arithmetic_mean_builtin, cmap='gray')
plt.title('Built-in Mean Filter')
plt.axis('off')

plt.subplot(133)
plt.imshow(arithmetic_mean_manual, cmap='gray')
plt.title('Manual Mean Filter')
plt.axis('off')

plt.tight_layout()
plt.show()


## **Geometric Means**

In [ ]:
def geometric_mean_filter(image, kernel_size=3):
    img_float = image.astype(np.float64) + 1e-6
    log_img = np.log(img_float)

    kernel = np.ones((kernel_size, kernel_size), np.float32) / (kernel_size * kernel_size)
    mean_log = cv2.filter2D(log_img, -1, kernel)

    geo_mean = np.exp(mean_log)
    return np.clip(geo_mean, 0, 255).astype(np.uint8)

noise_img = noise_gauss
geometric_result = geometric_mean_filter(noise_img, kernel_size=3)

plt.figure(figsize=(10, 5))

plt.subplot(121)
plt.imshow(noise_img, cmap='gray')
plt.title('Gaussian Noisy Image')
plt.axis('off')

plt.subplot(122)
plt.imshow(geometric_result, cmap='gray')
plt.title('Geometric Mean Result')
plt.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# Geometric Mean Filter applied to all noisy images

noisy_images = [
    ('Gaussian', noise_gauss),
    ('Salt & Pepper', noise_sp),
    ('Exponential', noise_expo),
    ('Uniform', noise_uni)
]

plt.figure(figsize=(12, 8))

for i, (name, noisy) in enumerate(noisy_images, start=1):
    result = geometric_mean_filter(noisy, kernel_size=3)

    plt.subplot(4, 2, 2*i - 1)
    plt.imshow(noisy, cmap='gray')
    plt.title(f'{name} Noise')
    plt.axis('off')

    plt.subplot(4, 2, 2*i)
    plt.imshow(result, cmap='gray')
    plt.title(f'{name} after Geometric Mean')
    plt.axis('off')

plt.tight_layout()
plt.show()


## **Harmonic Means Filter**

In [ ]:
def harmonic_mean_filter(image, kernel_size=3):
    img_float = image.astype(np.float64) + 1e-6
    reciprocal_img = 1.0 / img_float

    kernel = np.ones((kernel_size, kernel_size), np.float32)
    sum_reciprocal = cv2.filter2D(reciprocal_img, -1, kernel)

    n = kernel_size * kernel_size
    harmonic_mean = n / (sum_reciprocal + 1e-6)

    return np.clip(harmonic_mean, 0, 255).astype(np.uint8)

noise_img = noise_gauss
harmonic_result = harmonic_mean_filter(noise_img, kernel_size=3)

plt.figure(figsize=(10, 5))

plt.subplot(121)
plt.imshow(noise_img, cmap='gray')
plt.title('Gaussian Noisy Image')
plt.axis('off')

plt.subplot(122)
plt.imshow(harmonic_result, cmap='gray')
plt.title('Harmonic Mean Result')
plt.axis('off')

plt.tight_layout()
plt.show()


## **Contra - Harmonic Mean Filter**

In [ ]:
def contra_harmonic_mean(image, kernel_size=3, Q=0):
    img_float = image.astype(np.float64) + 1e-6

    kernel = np.ones((kernel_size, kernel_size), np.float32)

    numerator = np.power(img_float, Q + 1)
    denominator = np.power(img_float, Q)

    num_sum = cv2.filter2D(numerator, -1, kernel)
    den_sum = cv2.filter2D(denominator, -1, kernel)

    result = num_sum / (den_sum + 1e-6)
    return np.clip(result, 0, 255).astype(np.uint8)

noise_img = noise_sp

pepper_removed = contra_harmonic_mean(noise_img, kernel_size=3, Q=1.5)
salt_removed = contra_harmonic_mean(noise_img, kernel_size=3, Q=-1.5)

plt.figure(figsize=(15, 5))

plt.subplot(131)
plt.imshow(noise_img, cmap='gray')
plt.title('Salt & Pepper Noisy Image')
plt.axis('off')

plt.subplot(132)
plt.imshow(pepper_removed, cmap='gray')
plt.title('Q = 1.5 Removes Pepper')
plt.axis('off')

plt.subplot(133)
plt.imshow(salt_removed, cmap='gray')
plt.title('Q = -1.5 Removes Salt')
plt.axis('off')

plt.tight_layout()
plt.show()


## **Median Filter**

In [ ]:
# ========== ( Median Filter ) =================

noise_img = noise_sp

median_result = cv2.medianBlur(noise_img, 3)

plt.figure(figsize=(10, 5))

plt.subplot(121)
plt.imshow(noise_img, cmap='gray')
plt.title('Salt & Pepper Noisy Image')
plt.axis('off')

plt.subplot(122)
plt.imshow(median_result, cmap='gray')
plt.title('After Median Filter')
plt.axis('off')

plt.tight_layout()
plt.show()


## **Max and Min Filter**

In [ ]:
# ========== ( Max and Min Filter ) =================

noise_img = noise_sp

kernel = np.ones((3, 3), np.uint8)

max_filter = cv2.dilate(noise_img, kernel)
min_filter = cv2.erode(noise_img, kernel)

plt.figure(figsize=(15, 5))

plt.subplot(131)
plt.imshow(noise_img, cmap='gray')
plt.title('Salt & Pepper Noisy Image')
plt.axis('off')

plt.subplot(132)
plt.imshow(max_filter, cmap='gray')
plt.title('Max Filter - Removes Pepper')
plt.axis('off')

plt.subplot(133)
plt.imshow(min_filter, cmap='gray')
plt.title('Min Filter - Removes Salt')
plt.axis('off')

plt.tight_layout()
plt.show()


## **Mid-Point Filtering**

In [ ]:
def midpoint_filter(image, kernel_size=3):
    kernel = np.ones((kernel_size, kernel_size), np.uint8)

    max_img = cv2.dilate(image, kernel).astype(np.float64)
    min_img = cv2.erode(image, kernel).astype(np.float64)

    midpoint = (max_img + min_img) / 2.0
    return np.clip(midpoint, 0, 255).astype(np.uint8)

noise_img = noise_uni

midpoint_result = midpoint_filter(noise_img, kernel_size=3)

plt.figure(figsize=(10, 5))

plt.subplot(121)
plt.imshow(noise_img, cmap='gray')
plt.title('Uniform Noisy Image')
plt.axis('off')

plt.subplot(122)
plt.imshow(midpoint_result, cmap='gray')
plt.title('After Midpoint Filter')
plt.axis('off')

plt.tight_layout()
plt.show()
